# 8장 실습 — 수요 만들기

지금까지는 저장소에 들어 있던 수요를 그대로 썼습니다.
이번에는 직접 만듭니다. 기말 프로젝트에서 여러분의 동네 수요를 만들 때 쓰는 절차입니다.
교재 8장에 대응합니다.

만든 수요는 그 자리에서 11장의 시뮬레이션 루프에 넣어 결과가 어떻게 달라지는지 봅니다.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect, todo
from smartmob.viz import use_korean_font

use_korean_font()

## 1. 시뮬레이터가 받는 형식 (교재 8.4)

수요는 컬럼 다섯 개짜리 표입니다. 이 형식만 맞으면 어떻게 만들었든 상관없습니다.

In [ ]:
from smartmob.data import DEMAND_COLUMNS, load_demand, validate_demand

print("필수 컬럼:", DEMAND_COLUMNS)

demand = load_demand("hanam")
print(demand.shape)
demand.head()

`validate_demand` 가 형식을 검사합니다.
좌표를 (경도, 위도) 순으로 넣는 실수를 여기서 잡습니다.

In [ ]:
validate_demand(demand)
print("[v] 형식 통과")

## 2. 시간대 패턴 (교재 8.2)

In [ ]:
import matplotlib.pyplot as plt

hourly = demand["request_time"].floordiv(60).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(hourly.index, hourly.values, color="#4C6EF5")
ax.set_xlabel("시각 (시)")
ax.set_ylabel("호출 수")
ax.set_title("하남 택시 수요의 시간대 분포")
plt.tight_layout();

## 3. 1단계 — 경계 안에 균등하게 (교재 8.5)

가장 단순한 방법입니다. 시군구 경계 안에 점을 고르게 뿌립니다.

In [ ]:
from smartmob.data import load_sigungu
from smartmob.teaching.demand_gen import generate_demand

boundary = load_sigungu("하남시")
flat = generate_demand(boundary=boundary, n=1000, seed=42, hourly=None)

validate_demand(flat)
print(flat.shape)
flat.head(3)

문제가 있습니다. 산과 강 위에서도 호출이 생깁니다.
승객이 산 한가운데에서 택시를 부르면 시뮬레이터가 그 지점을 도로에 스냅하느라 엉뚱한 곳으로 보냅니다.

## 4. 2단계 — 도로 위에 (교재 8.6)

도로망의 엣지 위에서 점을 뽑으면 이 문제가 사라집니다.
엣지 길이에 비례해 뽑으므로 큰길 주변에 더 많이 생깁니다.

In [ ]:
from smartmob.data import load_road_graph

G = load_road_graph("hanam", modes=("drive",))
on_road = generate_demand(graph=G, n=1000, seed=42, hourly=None)

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5), sharex=True, sharey=True)
for ax, df, title in [(axes[0], flat, "경계 안 균등"), (axes[1], on_road, "도로 위")]:
    ax.scatter(df["origin_lon"], df["origin_lat"], s=4, alpha=0.4, color="#4C6EF5")
    ax.set_title(title)
    ax.set_xlabel("경도")
axes[0].set_ylabel("위도")
plt.tight_layout();

## 5. 3단계 — 시간대 프로파일 (교재 8.7)

지금까지는 저녁 시간에 호출이 고르게 흩어져 있었습니다.
실제 수요는 특정 시각에 몰립니다. `hourly` 로 시간대 비중을 줍니다.

In [ ]:
from smartmob.teaching.demand_gen import HANAM_HOURLY

realistic = generate_demand(graph=G, n=1000, seed=42, hourly=HANAM_HOURLY)

fig, ax = plt.subplots(figsize=(8, 3.5))
for df, label in [(on_road, "고르게"), (realistic, "실제 프로파일")]:
    counts = df["request_time"].floordiv(60).value_counts().sort_index()
    ax.plot(counts.index, counts.values, marker="o", label=label)
ax.set_xlabel("시각 (시)")
ax.set_ylabel("호출 수")
ax.legend()
plt.tight_layout();

## 6. 만든 수요로 시뮬레이션 돌리기

여기가 이 장의 핵심입니다. 수요를 바꾸면 결과가 어떻게 달라지는지 직접 봅니다.
11장의 루프를 미리 빌려 씁니다. 서버 없이 로컬에서 1초 안에 돕니다.

In [ ]:
from smartmob.data import load_vehicles
from smartmob.teaching.simloop import simulate

vehicles = load_vehicles("hanam")
print(f"차량 {len(vehicles)}대")

runs = {}
for label, df in [("고르게 흩뿌린 수요", on_road), ("실제 프로파일 수요", realistic)]:
    runs[label] = simulate(df, vehicles, 1080, 1440)

import pandas as pd

pd.DataFrame({
    label: {
        "서비스율": round(r.summary()["service_rate"], 3),
        "평균대기_분": round(r.summary()["avg_waiting_time_min"], 2),
        "최대대기_분": round(r.summary()["max_waiting_time_min"], 1),
        "가동률": round(r.summary()["utilization"], 3),
    }
    for label, r in runs.items()
}).T

같은 차량 대수, 같은 호출 건수인데 결과가 다릅니다.
수요가 몰리면 그 시간대에 차가 모자라기 때문입니다.
**총량이 아니라 분포가 서비스 수준을 정합니다.**

## 7. 빈칸

### 7.1 나만의 시간대 프로파일

24개짜리 리스트를 만들어 넣습니다. 합이 1이 아니어도 됩니다. 안에서 정규화합니다.
예를 들어 저녁 9시에 극단적으로 몰리는 프로파일을 만들어 보세요.

In [ ]:
my_hourly = None      # 길이 24의 리스트. 예: [0]*18 + [1, 5, 2, 1] + [0, 0]

banner("빈칸 7.1")
if my_hourly:
    peaky = generate_demand(graph=G, n=1000, seed=42, hourly=tuple(my_hourly))
    run = simulate(peaky, vehicles, 1080, 1440)
    s = run.summary()
    print(f"[v] 서비스율 {s['service_rate']:.1%}, "
          f"평균대기 {s['avg_waiting_time_min']:.2f}분, "
          f"최대대기 {s['max_waiting_time_min']:.1f}분")
else:
    print("[ ] my_hourly 를 채우세요")

### 7.2 서비스율이 무너지는 지점

호출 건수를 1,000건에서 늘려 가며 서비스율이 90% 아래로 떨어지는 지점을 찾습니다.
차량은 80대 그대로 둡니다.

In [ ]:
break_point = None      # 서비스율이 90% 아래로 내려가는 호출 건수

# 실험용 코드입니다. 값을 바꿔 가며 돌려 보세요.
for n in [1000, 2000, 3000]:
    df = generate_demand(graph=G, n=n, seed=42, hourly=HANAM_HOURLY)
    s = simulate(df, vehicles, 1080, 1440).summary()
    print(f"호출 {n:5,}건 → 서비스율 {s['service_rate']:.1%}, "
          f"평균대기 {s['avg_waiting_time_min']:.2f}분")

banner("빈칸 7.2")
todo("서비스율이 90% 아래로 내려가는 호출 건수", break_point)

### 7.3 seed 를 바꾸면

`seed` 만 바꿔 다섯 번 돌려 서비스율의 표준편차를 구합니다.
난수가 어디에 들어 있는지가 중요합니다. 루프 자체는 결정론적이고, 난수는 **수요 생성** 에 있습니다.

In [ ]:
service_rate_std = None     # seed 5개에서 나온 서비스율의 표준편차

banner("빈칸 7.3")
todo("서비스율 표준편차", service_rate_std, fmt=lambda v: f"{v:.4f}")

## 정리

- 수요는 컬럼 다섯 개짜리 표입니다. `validate_demand` 로 형식을 먼저 확인합니다
- 경계 안 균등 → 도로 위 → 시간대 프로파일 순으로 현실에 가까워집니다
- 총 호출 건수가 같아도 시간대 분포가 다르면 서비스 수준이 달라집니다
- 난수는 수요 생성에 있습니다. seed 를 고정하지 않으면 비교가 무의미해집니다
- 9장 실습에서는 배차에 필요한 도착 예상시간을 모델로 예측합니다